# MAZEWARD VERSUS — Colab 学習サーバー

ローカルの GUI から Colab の GPU 学習を **開始 / 停止 / 監視** するための一式です。

## 手順

| | セル | 何をするか |
|---|---|---|
| (1) | Setup | Drive をマウントしてコードを取り込む。**API トークンを Drive に固定** |
| (2) | 制御サーバー起動 | 学習を操作する Flask を立てて、生きているか確認 |
| (3) | ngrok 公開 | 外から届く URL を作る。**GUI に貼る値がここに出ます** |
| (4) | 死活監視 | 切断を防ぐ。**このセルは回したままにしてください** |
| (5) | 状況 / 停止 | 学習の様子を見る・止める |

## 以前との違い（つまずきの対策）

- **API トークンが毎回変わっていた** … `API.txt` を `/content` に作っていたため、
  セルを回すたび新しくなっていました。**Drive に置いて固定**したので、
  GUI へ貼り直すのは最初の 1 回だけです。
- **ngrok authtoken を毎回入力していた** … Colab シークレットか Drive から読みます。
- **URL が毎回変わる** … 固定ドメインがあれば Drive の `ngrok_domain.txt` に書くと同じ URL になります。
- **しばらくすると切れる** … (4) の死活監視セルを回しておくと、アイドル切断を避けつつ
  切断を画面で検知できます。GUI 側には `SSL: UNEXPECTED_EOF` や `ERR_NGROK_3200` として見えます。


## (1) Setup — コード取り込みと API トークンの固定


In [ ]:
import os, shutil, secrets, pathlib
from google.colab import drive

DRIVE_FOLDER = 'mazeward_colab_rl_ai'      # Drive 側のフォルダ名
COLAB_DIR    = '/content/mazeward_colab_rl_ai'
PORT         = 5558

drive.mount('/content/drive', force_remount=True)
DRIVE_ROOT = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
assert os.path.exists(DRIVE_ROOT), (
    f'Drive に {DRIVE_ROOT} がありません。'
    'ローカル GUI の「Colab・Drive連携 → コードを Colab へ送信」を先に実行してください')

# 毎回まっさらにしてから配置する（古いコードが残ると原因が分からなくなる）
if os.path.exists(COLAB_DIR):
    shutil.rmtree(COLAB_DIR)
shutil.copytree(DRIVE_ROOT, COLAB_DIR,
                ignore=shutil.ignore_patterns('models', 'replays', '__pycache__',
                                              '*.ipynb', 'API.txt', '*.pt'))

# ---- API トークンは Drive に置いて固定する ----
# 以前は /content に作っていたので、ランタイムが変わるたび新しい値になり、
# そのたび GUI へ貼り直す必要があった。Drive に置けば次回もそのまま使える。
drive_token = pathlib.Path(DRIVE_ROOT) / 'API.txt'
if not drive_token.exists():
    drive_token.write_text(secrets.token_hex(16), encoding='utf-8')
    print('API トークンを新規作成しました（次回以降は使い回します）')
API_TOKEN = drive_token.read_text(encoding='utf-8').strip()

local_token = pathlib.Path(COLAB_DIR) / 'ai' / 'API.txt'
local_token.parent.mkdir(parents=True, exist_ok=True)
local_token.write_text(API_TOKEN, encoding='utf-8')

print('コード配置:', COLAB_DIR)
print('API トークン:', API_TOKEN)
print('  ^ GUI の「API トークン」欄に貼ります（Drive 保存済みなので次回も同じ）')


## (2) 制御サーバーを起動 — 生きているか必ず確認する


In [ ]:
import subprocess, sys, time, json, urllib.request

LOG_PATH = '/content/mazeward_control.log'

# 既に動いていたら止める（二重起動するとポートを奪い合う）
subprocess.run(['pkill', '-f', 'mazeward_colab_control.py'], check=False)
time.sleep(1)

log = open(LOG_PATH, 'w', encoding='utf-8')
proc = subprocess.Popen(
    [sys.executable, f'{COLAB_DIR}/ai/mazeward_colab_control.py', '--port', str(PORT)],
    cwd=COLAB_DIR, stdout=log, stderr=subprocess.STDOUT)

# 立ち上がるまで待ち、**実際に応答するか**を確かめる。
# ここを省くと、死んでいるサーバーに ngrok を張って 404 で悩むことになる。
ok = False
for _ in range(30):
    time.sleep(1)
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/healthz', timeout=3) as r:
            ok = json.loads(r.read().decode()).get('ok', False)
        if ok:
            break
    except Exception:
        if proc.poll() is not None:
            break

if ok:
    print(f'制御サーバー起動 OK  pid={proc.pid}  port={PORT}')
else:
    print('起動に失敗しました。ログの末尾:')
    print(open(LOG_PATH, encoding='utf-8').read()[-2000:])
    raise SystemExit('制御サーバーが起動していません')


## (3) ngrok で公開 — ここに出る 2 つを GUI に貼ります

`NGROK_AUTHTOKEN` は次の順に探します。**一度 Drive に置けば次回から入力不要**です。

1. Colab のシークレット（左の鍵アイコン → `NGROK_AUTHTOKEN`）
2. Drive の `ngrok_authtoken.txt`
3. その場で入力（入力したら Drive に保存します）

固定ドメインを持っているなら Drive に `ngrok_domain.txt` を置くと **毎回同じ URL** になります。


In [ ]:
import os, pathlib, getpass, subprocess, sys, json, urllib.request
try:
    from pyngrok import ngrok
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok

DRIVE_ROOT  = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
auth_file   = pathlib.Path(DRIVE_ROOT) / 'ngrok_authtoken.txt'
domain_file = pathlib.Path(DRIVE_ROOT) / 'ngrok_domain.txt'

auth = os.environ.get('NGROK_AUTHTOKEN', '').strip()
if not auth:
    try:
        from google.colab import userdata
        auth = (userdata.get('NGROK_AUTHTOKEN') or '').strip()
    except Exception:
        auth = ''
if not auth and auth_file.exists():
    auth = auth_file.read_text(encoding='utf-8').strip()
if not auth:
    auth = getpass.getpass('ngrok authtoken を入力（次回のため Drive へ保存します）: ').strip()
    if auth:
        auth_file.write_text(auth, encoding='utf-8')
        print('Drive に保存しました:', auth_file)
assert auth, 'ngrok authtoken がありません'
ngrok.set_auth_token(auth)

domain = os.environ.get('NGROK_DOMAIN', '').strip()
if not domain and domain_file.exists():
    domain = domain_file.read_text(encoding='utf-8').strip()

ngrok.kill()                     # 古いトンネルが残ると URL が増えて混乱する
tunnel = ngrok.connect(PORT, domain=domain) if domain else ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')

# 公開 URL 越しに疎通確認。ngrok の警告ページを避けるヘッダが必須
req = urllib.request.Request(PUBLIC_URL + '/healthz', headers={
    'ngrok-skip-browser-warning': 'true',
    'X-API-Token': API_TOKEN,
})
try:
    with urllib.request.urlopen(req, timeout=15) as r:
        reachable = json.loads(r.read().decode()).get('ok', False)
except Exception as e:
    reachable = False
    print('公開 URL への疎通に失敗:', e)

print('=' * 68)
print('  GUI（Colab・Drive連携 タブ）に貼る値')
print('=' * 68)
print('  ngrok URL   :', PUBLIC_URL)
print('  API トークン:', API_TOKEN)
print('=' * 68)
print('  疎通:', 'OK' if reachable else 'NG（(2) のログを確認してください）')
if not domain:
    print()
    print('  ヒント: 固定ドメインがあるなら Drive に ngrok_domain.txt を作って')
    print('        中にドメイン名を書いておくと、URL が毎回同じになります。')


## (4) 死活監視 — **このセルは回したままにしてください**

Colab はアイドルだとランタイムを切ります。切れると ngrok も落ち、
GUI 側には `SSL: UNEXPECTED_EOF` や `ERR_NGROK_3200` として見えます。
このセルを走らせておくと、生存確認をしながら学習の進み具合も表示します。


In [ ]:
import time, json, urllib.request, datetime

def _get(path):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}' + path,
                                 headers={'X-API-Token': API_TOKEN})
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read().decode())

print('死活監視を開始します（止めるにはこのセルを中断）')
while True:
    now = datetime.datetime.now().strftime('%H:%M:%S')
    try:
        st = _get('/colab/status')
        live = st.get('live') or {}
        latest = st.get('latest') or {}
        gen = live.get('episode') if live.get('episode') is not None else latest.get('episode', '-')
        step = f"{live.get('step')}/{live.get('max_steps')}" if live.get('step') else '-'
        print(f'[{now}] 学習中={st.get("is_running")} 世代={gen} '
              f'ステップ={step} 記録={st.get("gen_count")}世代', flush=True)
    except Exception as e:
        print(f'[{now}] 制御サーバーに繋がりません: {e}', flush=True)
    time.sleep(30)


## (5) 状況の確認 / 学習の停止（必要なときだけ）


In [ ]:
# --- いまの状況 ---
st = _get('/colab/status')
print(json.dumps({k: v for k, v in st.items() if k != 'logs'},
                 ensure_ascii=False, indent=2))
print()
print('--- 直近のログ ---')
for line in (st.get('logs') or [])[-15:]:
    print(f"{line.get('time')} [{line.get('tag')}] {line.get('text')}")


In [ ]:
# --- 学習を止める（GUI の「停止」と同じ） ---
import urllib.request, json
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/colab/stop', data=b'{}',
                             method='POST',
                             headers={'Content-Type': 'application/json',
                                      'X-API-Token': API_TOKEN})
with urllib.request.urlopen(req, timeout=20) as r:
    print(json.loads(r.read().decode()))


## 学習ログはどこに残るか

`trainer_pb.py` が書き込みのたびに Drive の `mazeward_checkpoints/` へコピーします。
**ランタイムが切れてもグラフの元データは消えません。**
ローカル GUI の「学習ログを取得」で `colab_data/` に取り込めます。
